# 02 — Phase-1 Baseline Training — **YOLOPX vehicle + lane baseline**

This notebook trains the YOLOPX-derived stage-1 baseline with only two tasks:
object detection and lane segmentation. The drivable-area branch is removed.
The default config is `YOLOPX`, using the anchor-free YOLOX detection head
and the C2-assisted YOLOPX lane head.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q yacs tqdm opencv-python-headless tensorboard

Mounted at /content/drive


In [2]:
import os, sys
REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

In [3]:
import torch
import torch.nn as nn
import numpy as np
import logging
import math
from pathlib import Path
from torch import amp  # torch.amp replaces torch.cuda.amp in recent PyTorch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as T

from lib.config import cfg
from lib.models import get_net
from lib.core import get_loss, train, validate
from lib.dataset import BddDataset

# Run mode — stage1 has two sanctioned modes:
#   'smoke' : 16-sample subset, 2 epochs, val plots off. For fast debug.
#   'full'  : full BDD100K, cfg.TRAIN.END_EPOCH epochs, val plots at epoch 0 only.
# The selected mode is applied at dataset-build + train-loop time; nothing
# about the model / loss / optimizer is changed between modes.
RUN_MODE = 'full'    # 'smoke' | 'full'

# Speed knobs — safe across CUDA versions, do not change baseline semantics.
# - TF32: ~2x matmul throughput on A5000/A100 with negligible accuracy loss.
# - cudnn.benchmark: picks the fastest conv algorithm for fixed-shape batches
#   (we letterbox to a fixed HxW per-split, so this is safe).
# - channels_last is an OPTIONAL toggle because small spatial heads in this
#   model (lane stride-1 decoder) see uneven speedup; leave it off by default.
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'RUN_MODE = {RUN_MODE}')

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
RUN_MODE = full


In [4]:

# ── Config selection + dataset roots ──
# Stage1 YOLOPX baseline. Flip CONFIG manually for comparison rows.
CONFIG = 'YOLOPX'   # 'YOLOPX' | 'YOLOP' | 'YOLOPv2-paper-no-da' | 'YOLOPv2-best-row' | 'YOLOPv2-focal-only'
GPU_PROFILE = 'G4_RTX_PRO_6000'   # 'none' | 'G4_RTX_PRO_6000'
BATCH_OVERRIDE = None             # e.g. 24 to force a run; None keeps YAML
RESUME_CKPT_OVERRIDE = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx/best.pth'        # e.g. '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx/best.pth'
RESUME_WEIGHTS_ONLY = True         # True = load model weights only and restart optimizer/scheduler with stable LR

from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive,
    find_raw_bdd_root,
    resolve_bdd_images_100k_dir,
    resolve_bdd_labels_100k_dir,
)

yaml_map = {
    'YOLOPv2-paper-no-da': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_paper_no_da.yaml'),
    'YOLOPv2-best-row':   os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_best_row.yaml'),
    'YOLOPv2-focal-only': os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopv2_focal_only_ablation.yaml'),
    'YOLOPX':             os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolopx_vehicle_lane_baseline.yaml'),
    'YOLOP':              os.path.join(REPO_ROOT, 'stage1', 'configs', 'yolop_vehicle_lane_baseline.yaml'),
}
run_name = {
    'YOLOPv2-paper-no-da': 'yolopv2_paper_no_da',
    'YOLOPv2-best-row':   'yolopv2_best_row',
    'YOLOPv2-focal-only': 'yolopv2_focal_only',
    'YOLOPX':             'yolopx',
    'YOLOP':              'yolop',
}[CONFIG]

cfg.defrost()
cfg.merge_from_file(yaml_map[CONFIG])

# Resolve dataset roots (Colab SSD -> Drive fallback).
ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
DATASET_ROOT = ensure_local_dataset_from_drive('bdd100k_vehicle5', ECOCAR_ROOT)
RAW_BDD_ROOT = find_raw_bdd_root(ECOCAR_ROOT)
BDD_IMAGES = resolve_bdd_images_100k_dir(RAW_BDD_ROOT, ECOCAR_ROOT)
BDD_LABELS = resolve_bdd_labels_100k_dir(RAW_BDD_ROOT)

cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = os.path.join(DATASET_ROOT, 'masks')

cfg.DRIVE.ROOT = ECOCAR_ROOT
cfg.DRIVE.CHECKPOINT_DIR = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'stage1', 'checkpoints', run_name)
cfg.DRIVE.METRICS_DIR    = os.path.join(ECOCAR_ROOT, 'yolop_vehicle_lane', 'stage1', 'metrics',     run_name)

# RTX PRO 6000 stable runtime profile.
# The YAML now uses conservative LR / augmentation / grad clipping because the
# previous high-LR run diverged late in training around epoch 68-69.
# Keep batch at 32 unless the loss curve is proven stable for at least 10 epochs.
if BATCH_OVERRIDE is not None:
    cfg.TRAIN.BATCH_SIZE_PER_GPU = int(BATCH_OVERRIDE)
    cfg.TEST.BATCH_SIZE_PER_GPU = int(BATCH_OVERRIDE)

if GPU_PROFILE == 'G4_RTX_PRO_6000':
    cfg.WORKERS = min(int(cfg.WORKERS), 4)
    cfg.PRINT_FREQ = 20

cfg.TEST.PLOTS = False
if RUN_MODE == 'smoke':
    cfg.TRAIN.END_EPOCH = 2
    cfg.TRAIN.VAL_FREQ = 1
    cfg.TEST.PLOTS = True

cfg.freeze()

os.makedirs(cfg.DRIVE.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(cfg.DRIVE.METRICS_DIR, exist_ok=True)
output_dir = cfg.DRIVE.METRICS_DIR
tb_log_dir = os.path.join(cfg.DRIVE.ROOT, 'yolop_vehicle_lane', 'stage1', 'tb_logs', run_name)
os.makedirs(tb_log_dir, exist_ok=True)

print(f'Config       : {CONFIG}')
print(f'YAML         : {yaml_map[CONFIG]}')
print(f'GPU profile  : {GPU_PROFILE}')
print(f'Model.NAME   : {cfg.MODEL.NAME}')
print(f'NC           : {cfg.MODEL.NC}  (classes: {cfg.MODEL.VEHICLE_CLASSES})')
print(f'Optimizer    : {cfg.TRAIN.OPTIMIZER}  LR0={cfg.TRAIN.LR0}  WD={cfg.TRAIN.WD}')
print(f'Batch(train) : {cfg.TRAIN.BATCH_SIZE_PER_GPU}   Batch(val): {cfg.TEST.BATCH_SIZE_PER_GPU}')
print(f'Warmup       : {cfg.TRAIN.WARMUP_EPOCHS}   GradClip: {getattr(cfg.TRAIN, "GRAD_CLIP_NORM", 0.0)}')
print(f'Focal γ      : cls/obj={cfg.LOSS.FL_GAMMA}  lane={getattr(cfg.LOSS, "LL_FL_GAMMA", 0.0)}')
print(f'YOLOPX loss  : det=0.02*(5*IoU+obj_focal), lane=0.2*focalBCE+0.2*Tversky')
print(f'Mosaic/MixUp : {getattr(cfg.DATASET, "MOSAIC", False)} / {getattr(cfg.DATASET, "MIXUP", False)}')
print(f'Augment      : scale={cfg.DATASET.SCALE_FACTOR} rot={cfg.DATASET.ROT_FACTOR} translate={cfg.DATASET.TRANSLATE}')
print(f'Val plots    : {cfg.TEST.PLOTS}')
print(f'Val freq     : every {cfg.TRAIN.VAL_FREQ} completed epoch(s)')
print(f'Checkpoints  : {cfg.DRIVE.CHECKPOINT_DIR}')


Extracting /content/drive/MyDrive/EcoCAR/datasets/bdd100k_vehicle5.tar.gz into this notebook runtime ...
[BDD images] selected: /content/bdd100k_raw/100k | train=70000, val=10000
Config       : YOLOPX
YAML         : /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/configs/yolopx_vehicle_lane_baseline.yaml
GPU profile  : G4_RTX_PRO_6000
Model.NAME   : YOLOPX
NC           : 1  (classes: ['vehicle'])
Optimizer    : adamw  LR0=0.0003  WD=0.0005
Batch(train) : 32   Batch(val): 32
Warmup       : 8.0   GradClip: 5.0
Focal γ      : cls/obj=2.0  lane=2.0
YOLOPX loss  : det=0.02*(5*IoU+obj_focal), lane=0.2*focalBCE+0.2*Tversky
Mosaic/MixUp : False / False
Augment      : scale=0.15 rot=5 translate=0.05
Val plots    : False
Val freq     : every 1 completed epoch(s)
Checkpoints  : /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx


In [5]:
# ── Logger ──
logger = logging.getLogger('train')
logger.setLevel(logging.INFO)
if not logger.handlers:
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    logger.addHandler(ch)

In [6]:
# ── Build datasets ──
# YOLOPv2 paper §3: train 640×640 letterbox, test 640×384 letterbox.
# MODEL.IMAGE_SIZE is written (W, H) in the YAML. BddDataset accepts
# either an int (square auto-letterbox) or an explicit (H, W) tuple
# (rectangular letterbox with auto=False).
#
# Long-run hygiene (REPAIR v5):
#   * val plots are expensive and are the biggest non-science cost once
#     training is working. We enable them for epoch 0 only as an alignment
#     snapshot. If the user wants every epoch's plots, they can flip
#     `cfg.TEST.PLOTS = True` between val calls.
#   * persistent_workers is intentionally OFF for Colab stability. It costs a
#     small epoch-boundary overhead but avoids native worker/kernel crashes.
transform = T.ToTensor()

train_wh = tuple(cfg.MODEL.IMAGE_SIZE)              # (W, H)
val_wh = tuple(getattr(cfg.TEST, 'IMAGE_SIZE', cfg.MODEL.IMAGE_SIZE))

train_size = int(max(train_wh))                     # 640 — square aug-friendly
val_size = (int(val_wh[1]), int(val_wh[0]))         # (H, W) for letterbox auto=False

train_dataset = BddDataset(cfg, is_train=True,  inputsize=train_size, transform=transform)
val_dataset   = BddDataset(cfg, is_train=False, inputsize=val_size,   transform=transform)

# Smoke mode applies a fixed-seed tiny subset so we can detect real learning
# signal in 1-2 minutes instead of burning a GPU hour. The SAME samples are
# used every iteration so the overfit-a-tiny-subset check is meaningful.
if RUN_MODE == 'smoke':
    import random as _rnd
    _g = _rnd.Random(1234)
    idx = sorted(_g.sample(range(len(train_dataset)), 16))
    train_dataset.db = [train_dataset.db[i] for i in idx]
    idx_v = sorted(_g.sample(range(len(val_dataset)), 16))
    val_dataset.db = [val_dataset.db[i] for i in idx_v]
    print(f'[SMOKE] reduced train→{len(train_dataset)} val→{len(val_dataset)}')

train_loader = DataLoader(
    train_dataset, batch_size=cfg.TRAIN.BATCH_SIZE_PER_GPU,
    shuffle=cfg.TRAIN.SHUFFLE, num_workers=cfg.WORKERS,
    pin_memory=cfg.PIN_MEMORY, collate_fn=train_dataset.collate_fn,
    persistent_workers=False,
    prefetch_factor=2 if cfg.WORKERS > 0 else None,
)
val_loader = DataLoader(
    val_dataset, batch_size=cfg.TEST.BATCH_SIZE_PER_GPU,
    shuffle=False, num_workers=cfg.WORKERS,
    pin_memory=cfg.PIN_MEMORY, collate_fn=val_dataset.collate_fn,
    persistent_workers=False,
    prefetch_factor=2 if cfg.WORKERS > 0 else None,
)

# Sanity check — one sample tells us the actual letterboxed HxW used at train vs eval.
if len(train_dataset) == 0 or len(val_dataset) == 0:
    raise RuntimeError(
        f'Empty dataset detected: train={len(train_dataset)}, val={len(val_dataset)}. '
        f'Check image root={cfg.DATASET.DATAROOT}, label root={cfg.DATASET.LABELROOT}, lane root={cfg.DATASET.LANEROOT}'
    )

sample_train = train_dataset[0][0].shape
sample_val = val_dataset[0][0].shape
print(f'Train: {len(train_dataset)} samples, letterbox C×H×W = {tuple(sample_train)}')
print(f'Val:   {len(val_dataset)} samples, letterbox C×H×W = {tuple(sample_val)}')


[Dataset] split=train | layout=explicit_packaged_like
[Dataset] images=/content/bdd100k_raw/100k/train
[Dataset] labels=/content/bdd100k_raw/100k/train
[Dataset] lanes =/content/bdd100k_vehicle5/masks/train
building database...


100%|██████████| 70000/70000 [00:06<00:00, 10368.27it/s]


database build finish: 70000 samples
missing lane masks skipped: 0
missing detection labels: 0
[Dataset] split=val | layout=explicit_packaged_like
[Dataset] images=/content/bdd100k_raw/100k/val
[Dataset] labels=/content/bdd100k_raw/100k/val
[Dataset] lanes =/content/bdd100k_vehicle5/masks/val
building database...


100%|██████████| 10000/10000 [00:00<00:00, 10296.87it/s]


database build finish: 10000 samples
missing lane masks skipped: 0
missing detection labels: 0
Train: 70000 samples, letterbox C×H×W = (3, 384, 640)
Val:   10000 samples, letterbox C×H×W = (3, 384, 640)


In [7]:
# ── Build model, loss, optimizer, scheduler ──
# Scheduler policy
# ----------------
# YOLOPX upstream uses the YOLOP-style LambdaLR cosine schedule after
# iter-based linear warmup.
# `TRAIN.SGDR` remains available for ablation, but the default YOLOPX
# baseline stays with LambdaLR because that is what the public training
# script uses.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# get_net reads cfg.MODEL.NC and builds Detect with the right class count
# and sets model.names from the class-protocol registry. Do NOT override
# them here — doing so breaks the detector head / NMS label indexing
# (that was the KeyError bug in the stage1-1c run).
model = get_net(cfg).to(device)
model.gr = 1.0

criterion = get_loss(cfg, device)

if cfg.TRAIN.OPTIMIZER == 'adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.TRAIN.LR0,
                                 betas=(cfg.TRAIN.MOMENTUM, 0.999))
elif cfg.TRAIN.OPTIMIZER == 'adamw':
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.TRAIN.LR0,
                                  betas=(cfg.TRAIN.MOMENTUM, 0.999))
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.TRAIN.LR0,
                                momentum=cfg.TRAIN.MOMENTUM,
                                weight_decay=cfg.TRAIN.WD,
                                nesterov=cfg.TRAIN.NESTEROV)
for pg in optimizer.param_groups:
    pg['initial_lr'] = cfg.TRAIN.LR0

use_sgdr = bool(getattr(cfg.TRAIN, 'SGDR', False))
if use_sgdr:
    sgdr_t0 = int(getattr(cfg.TRAIN, 'SGDR_T0', max(1, cfg.TRAIN.END_EPOCH // 3)))
    sgdr_tmult = int(getattr(cfg.TRAIN, 'SGDR_TMULT', 1))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=sgdr_t0, T_mult=sgdr_tmult,
        eta_min=cfg.TRAIN.LR0 * cfg.TRAIN.LRF,
    )
    sched_desc = f'SGDR (T_0={sgdr_t0}, T_mult={sgdr_tmult})'
else:
    lf = lambda x: ((1 + math.cos(x * math.pi / cfg.TRAIN.END_EPOCH)) / 2) * \
                   (1 - cfg.TRAIN.LRF) + cfg.TRAIN.LRF
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)
    sched_desc = 'cosine-annealing + linear warmup (YOLOP default)'

scaler = amp.GradScaler(device.type, enabled=device.type != 'cpu')

num_params = sum(p.numel() for p in model.parameters())
print(f'Model nc/names: {model.nc}  {model.names}')
print(f'Model params : {num_params/1e6:.2f}M')
print(f'Device       : {device}')
print(f'Optimizer    : {cfg.TRAIN.OPTIMIZER} @ LR0={cfg.TRAIN.LR0}, WD={cfg.TRAIN.WD}')
print(f'Scheduler    : {sched_desc}')
print(f'Warmup       : {cfg.TRAIN.WARMUP_EPOCHS} epochs (iter-based linear in-loop)')

Model nc/names: 1  ['vehicle']
Model params : 31.38M
Device       : cuda
Optimizer    : adamw @ LR0=0.0003, WD=0.0005
Scheduler    : cosine-annealing + linear warmup (YOLOP default)
Warmup       : 8.0 epochs (iter-based linear in-loop)


In [8]:
# Optional speed probe. Set RUN_SPEED_PROBE = True only when diagnosing slow batches.
RUN_SPEED_PROBE = False
if RUN_SPEED_PROBE:
    import time
    model.train()
    probe_iter = iter(train_loader)
    for k in range(8):
        input, target, paths, shapes = next(probe_iter)
        t0 = time.perf_counter()
        input = input.to(device, non_blocking=True)
        target = [t.to(device, non_blocking=True) for t in target]
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        with amp.autocast(device_type=device.type, enabled=device.type != 'cpu'):
            outputs = model(input)
            total_loss, head_losses = criterion(outputs, target, shapes, model)
        torch.cuda.synchronize()
        t2 = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()
        t3 = time.perf_counter()
        print(f'probe {k}: move={t1-t0:.3f}s forward+loss={t2-t1:.3f}s backward+step={t3-t2:.3f}s total={t3-t0:.3f}s')


In [9]:
# ── Resume from checkpoint if available ──
# Recovery rule:
#   * For a collapsed run, do NOT resume from latest.pth, because latest may already contain bad weights.
#   * Use RESUME_CKPT_OVERRIDE='.../best.pth' or '.../best_joint.pth'.
#   * RESUME_WEIGHTS_ONLY=True loads only model weights and restarts optimizer/scheduler with the new stable LR.
start_epoch = cfg.TRAIN.BEGIN_EPOCH
best_map = 0.0
best_ll_iou = 0.0
best_joint = 0.0

manual_ckpt = RESUME_CKPT_OVERRIDE
auto_ckpt = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'latest.pth')
ckpt_path = manual_ckpt if manual_ckpt else auto_ckpt

if os.path.exists(ckpt_path) and (manual_ckpt is not None or cfg.AUTO_RESUME):
    ckpt = torch.load(ckpt_path, map_location=device)

    try:
        model.load_state_dict(ckpt['state_dict'])
    except RuntimeError as e:
        print('[warn] resume failed due to shape mismatch:', e)
        print('       This usually means the checkpoint was trained with a different model/class schema.')
        raise

    if not RESUME_WEIGHTS_ONLY:
        if 'optimizer' in ckpt and ckpt['optimizer'] is not None:
            optimizer.load_state_dict(ckpt['optimizer'])
        if 'scheduler' in ckpt and ckpt['scheduler'] is not None and scheduler is not None:
            scheduler.load_state_dict(ckpt['scheduler'])
        if 'scaler' in ckpt and ckpt['scaler'] is not None and scaler is not None:
            scaler.load_state_dict(ckpt['scaler'])
        start_epoch = int(ckpt.get('epoch', -1)) + 1
    else:
        completed = int(ckpt.get('completed_epoch', ckpt.get('epoch', -1) + 1))
        start_epoch = completed
        for pg in optimizer.param_groups:
            pg['lr'] = cfg.TRAIN.LR0
            pg['initial_lr'] = cfg.TRAIN.LR0
        print('[resume] Loaded model weights only. Optimizer/scheduler/scaler were reset to stable settings.')

    best_map = float(ckpt.get('best_map', ckpt.get('mAP50', 0.0)))
    best_ll_iou = float(ckpt.get('best_ll_iou', ckpt.get('ll_iou', 0.0)))
    best_joint = float(ckpt.get('best_joint', ckpt.get('joint_score', 0.0)))

    print(f'Resumed from: {ckpt_path}')
    print(f'start_epoch={start_epoch}, best_map={best_map:.4f}, best_ll_iou={best_ll_iou:.4f}, best_joint={best_joint:.4f}')
else:
    print('Training from scratch')


[resume] Loaded model weights only. Optimizer/scheduler/scaler were reset to stable settings.
Resumed from: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/checkpoints/yolopx/best.pth
start_epoch=66, best_map=0.8123, best_ll_iou=0.1943, best_joint=0.5016


In [ ]:
# ── Training loop ──
# Checkpoint-selection policy:
#   * latest.pth      — latest safe checkpoint only; if validation collapses it is restored from best.pth
#   * latest_train.pth — raw checkpoint immediately after training epoch, before validation
#   * unstable_epoch_XXXX.pth — saved if collapse is detected, for debugging only
#   * epoch_XXXX.pth  — saved for the first 3 epochs and then every SAVE_EPOCH_INTERVAL epochs
#   * best_det.pth    — best validation mAP50
#   * best_lane.pth   — best validation lane IoU
#   * best_joint.pth  — best validation joint score, also mirrored as best.pth
writer = SummaryWriter(tb_log_dir)
writer_dict = {
    'writer': writer,
    'train_global_steps': start_epoch * len(train_loader),
}

num_batch = len(train_loader)
num_warmup = max(round(cfg.TRAIN.WARMUP_EPOCHS * num_batch), 1000)

SAVE_EPOCH_INTERVAL = 5
STOP_ON_COLLAPSE = True


def _save_ckpt(name: str, state: dict):
    path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, name)
    torch.save(state, path)
    logger.info(f'Saved checkpoint: {path}')
    return path


def _load_ckpt(path: str):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['state_dict'])
    if 'optimizer' in ckpt and ckpt['optimizer'] is not None:
        optimizer.load_state_dict(ckpt['optimizer'])
    if 'scheduler' in ckpt and ckpt['scheduler'] is not None and scheduler is not None:
        scheduler.load_state_dict(ckpt['scheduler'])
    if 'scaler' in ckpt and ckpt['scaler'] is not None and scaler is not None:
        scaler.load_state_dict(ckpt['scaler'])
    return ckpt


def _set_plots(enabled: bool):
    cfg.defrost()
    cfg.TEST.PLOTS = bool(enabled)
    cfg.freeze()


def _build_state(epoch: int, completed_epoch: int, metrics: dict | None = None):
    state = {
        'epoch': epoch,
        'completed_epoch': completed_epoch,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler is not None else None,
        'scaler': scaler.state_dict() if scaler is not None else None,
        'best_map': best_map,
        'best_ll_iou': best_ll_iou,
        'best_joint': best_joint,
        'lr0': float(cfg.TRAIN.LR0),
        'batch_size': int(cfg.TRAIN.BATCH_SIZE_PER_GPU),
    }
    if metrics:
        state.update(metrics)
    return state


def _is_metric_collapse(completed_epoch: int, map50: float, ll_iou: float, joint: float, val_loss: float):
    if completed_epoch < 3:
        return False, ''
    if not np.isfinite(val_loss) or not np.isfinite(map50) or not np.isfinite(ll_iou) or not np.isfinite(joint):
        return True, 'non-finite validation metric'
    if best_joint >= 0.05 and joint < 0.20 * best_joint:
        return True, f'joint dropped from best {best_joint:.4f} to {joint:.4f}'
    if best_map >= 0.05 and best_ll_iou >= 0.05:
        det_bad = map50 < max(0.001, 0.10 * best_map)
        lane_bad = ll_iou < max(0.01, 0.10 * best_ll_iou)
        if det_bad and lane_bad:
            return True, f'both tasks collapsed: mAP50={map50:.4f}, LL_IoU={ll_iou:.4f}'
    return False, ''


import json as _json_per_epoch

PER_EPOCH_JSON_PATH = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane/stage1/metrics/yolopx/per_epoch.json'

def _persist_per_epoch_metrics(completed_epoch: int, metrics: dict) -> None:
    """Append one row to stage1/metrics/yolopx/per_epoch.json so Stage 1 lane
    metrics (LL_IoU, LL_Acc, mAP50) can be plotted per-epoch alongside Stage 2.
    Existing rows are preserved; rows for the same completed_epoch are
    overwritten so resumed runs do not duplicate."""
    import os as _os_per_epoch
    _os_per_epoch.makedirs(_os_per_epoch.path.dirname(PER_EPOCH_JSON_PATH), exist_ok=True)
    rows = []
    if _os_per_epoch.path.exists(PER_EPOCH_JSON_PATH):
        try:
            with open(PER_EPOCH_JSON_PATH, encoding='utf-8') as _fh:
                rows = _json_per_epoch.load(_fh)
            if not isinstance(rows, list):
                rows = []
        except Exception:
            rows = []
    rows = [r for r in rows if int(r.get('completed_epoch', -1)) != int(completed_epoch)]
    row = {'completed_epoch': int(completed_epoch)}
    row.update({k: float(v) for k, v in metrics.items()})
    rows.append(row)
    rows.sort(key=lambda r: int(r.get('completed_epoch', -1)))
    with open(PER_EPOCH_JSON_PATH, 'w', encoding='utf-8') as _fh:
        _json_per_epoch.dump(rows, _fh, indent=2)
    print(f'[per_epoch] wrote {PER_EPOCH_JSON_PATH} ({len(rows)} rows)', flush=True)


for epoch in range(start_epoch, cfg.TRAIN.END_EPOCH):
    train(cfg, train_loader, model, criterion, optimizer, scaler,
          epoch, num_batch, num_warmup, writer_dict, logger, device)

    scheduler.step()

    completed_epoch = epoch + 1

    train_state = _build_state(epoch, completed_epoch)
    _save_ckpt('latest_train.pth', train_state)

    if completed_epoch <= 3 or completed_epoch % SAVE_EPOCH_INTERVAL == 0:
        _save_ckpt(f'epoch_{completed_epoch:04d}.pth', train_state)

    do_val = (completed_epoch % int(cfg.TRAIN.VAL_FREQ) == 0)
    if do_val:
        logger.info(f'Running validation after completed epoch {completed_epoch}...')

        if RUN_MODE == 'full':
            _set_plots(completed_epoch == 1)

        ll_seg_result, det_result, val_loss, maps, times = validate(
            epoch, cfg, val_loader, val_dataset, model, criterion,
            output_dir, tb_log_dir, writer_dict, logger, device
        )

        ll_acc, ll_iou, ll_miou = ll_seg_result
        mp, mr, map50, map_all = det_result
        joint = 0.5 * float(map50) + 0.5 * float(ll_iou)

        writer.add_scalar('val/loss', val_loss, completed_epoch)
        writer.add_scalar('val/mAP50', map50, completed_epoch)
        writer.add_scalar('val/mAP', map_all, completed_epoch)
        writer.add_scalar('val/ll_iou', ll_iou, completed_epoch)
        writer.add_scalar('val/ll_acc', ll_acc, completed_epoch)
        writer.add_scalar('val/joint_score', joint, completed_epoch)

        logger.info(
            f'Validation after epoch {completed_epoch} | mAP50={map50:.4f} mAP={map_all:.4f} | '
            f'LL_IoU={ll_iou:.4f} LL_Acc={ll_acc:.4f} | '
            f'Joint={joint:.4f} | Loss={val_loss:.4f}'
        )

        metrics = {
            'joint_score': float(joint),
            'mAP50': float(map50),
            'mAP': float(map_all),
            'll_iou': float(ll_iou),
            'll_acc': float(ll_acc),
            'val_loss': float(val_loss),
        }
        _persist_per_epoch_metrics(completed_epoch, metrics)

        collapsed, reason = _is_metric_collapse(completed_epoch, float(map50), float(ll_iou), float(joint), float(val_loss))
        if collapsed:
            logger.warning(f'Validation collapse detected after epoch {completed_epoch}: {reason}')
            _save_ckpt(f'unstable_epoch_{completed_epoch:04d}.pth', _build_state(epoch, completed_epoch, metrics))
            recovery_path = os.path.join(cfg.DRIVE.CHECKPOINT_DIR, 'best.pth')
            if os.path.exists(recovery_path):
                recovered = _load_ckpt(recovery_path)
                logger.warning(f'Restored model/optimizer from best.pth at completed_epoch={recovered.get("completed_epoch", recovered.get("epoch", "unknown"))}')
                _save_ckpt('latest.pth', recovered)
            else:
                logger.warning('No best.pth exists yet; latest.pth was not overwritten by the unstable checkpoint.')
            if STOP_ON_COLLAPSE:
                logger.warning('Stopping training to protect the run. Restart from scratch with the stable YAML, or resume from best.pth manually after inspection.')
                break

        if map50 > best_map:
            best_map = float(map50)
            state = _build_state(epoch, completed_epoch, metrics)
            _save_ckpt('best_det.pth', state)
            logger.info(f'  ** new best_det: mAP50={best_map:.4f}')

        if ll_iou > best_ll_iou:
            best_ll_iou = float(ll_iou)
            state = _build_state(epoch, completed_epoch, metrics)
            _save_ckpt('best_lane.pth', state)
            logger.info(f'  ** new best_lane: LL_IoU={best_ll_iou:.4f}')

        if joint > best_joint:
            best_joint = float(joint)
            state = _build_state(epoch, completed_epoch, metrics)
            _save_ckpt('best_joint.pth', state)
            _save_ckpt('best.pth', state)
            logger.info(f'  ** new best_joint: {best_joint:.4f}')

        _save_ckpt('latest.pth', _build_state(epoch, completed_epoch, metrics))
    else:
        _save_ckpt('latest.pth', train_state)
        logger.info(
            f'Skipped validation after epoch {completed_epoch}; '
            f'next validation is every {cfg.TRAIN.VAL_FREQ} completed epoch(s).'
        )

writer.close()
print(f'\nTraining complete. best_det mAP50={best_map:.4f} | '
      f'best_lane LL_IoU={best_ll_iou:.4f} | best_joint={best_joint:.4f}')
